In [9]:
from catboost.datasets import titanic
import numpy as np

df_train, df_test = titanic()
df_train[:5]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
null_value_stats = df_train.isnull().sum(axis=0)
null_value_stats[null_value_stats != 0]

Age         177
Cabin       687
Embarked      2
dtype: int64

Заполним пропуски в данных некоторым уникальным значением

In [11]:
df_train.fillna(-999, inplace=True)
df_test.fillna(-999, inplace=True)

In [12]:
x = df_train.drop('Survived', axis=1)
y = df_train['Survived']

x.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [13]:
categorial_features = np.where(x.dtypes != float)[0]

In [14]:
from sklearn.model_selection import train_test_split

xtrain, xval, ytrain, yval = train_test_split(x, y, test_size=0.25, random_state=42)
xtest = df_test

In [15]:
from catboost import CatBoostClassifier, Pool, metrics, cv
from sklearn.metrics import accuracy_score

model = CatBoostClassifier(custom_loss=[metrics.Accuracy()],
                           random_seed=42)

In [16]:
model.fit(xtrain, ytrain, cat_features=categorial_features, # номера колонок с категориальными значениями
          eval_set=(xval, yval),  # датасет для оценки качества
          logging_level='Silent', plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

CatBoostClassifier(custom_loss=['Accuracy'], random_seed=42)

Кросс-валидация

In [17]:
cv_params = model.get_params()
cv_params.update({
    'loss_function': metrics.Logloss()
})
cv_data = cv(
    Pool(x, y, cat_features=categorial_features),
    cv_params,
    logging_level='Silent',
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Посмотрим на среднее качество и разброс по кросс-валидации

In [19]:
print('Best validation accuracy score: {:.2f}±{:.2f} on step {}'.format(
    np.max(cv_data['test-Accuracy-mean']),
    cv_data['test-Accuracy-std'][np.argmax(cv_data['test-Accuracy-mean'])],
    np.argmax(cv_data['test-Accuracy-mean'])
))

Best validation accuracy score: 0.83±0.02 on step 355


In [18]:
pred = model.predict(xtest)
pred_probs = model.predict_proba(xtest)
print(pred[:10])
print(pred_probs[:10])

[0 0 0 0 1 0 1 0 1 0]
[[0.85473931 0.14526069]
 [0.76313031 0.23686969]
 [0.88972889 0.11027111]
 [0.87876173 0.12123827]
 [0.3611047  0.6388953 ]
 [0.90513381 0.09486619]
 [0.33434185 0.66565815]
 [0.78468564 0.21531436]
 [0.39429048 0.60570952]
 [0.94047549 0.05952451]]


Ранняя остановка - детектор переобучения

In [20]:
model.fit(
    xtrain, ytrain,
    cat_features=categorial_features,
    eval_set=(xval, yval),
    early_stopping_rounds = 30,
    logging_level='Silent',
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

CatBoostClassifier(custom_loss=['Accuracy'], random_seed=42)

In [21]:
model.tree_count_

284

Важность признаков

In [22]:
feature_importances = model.get_feature_importance()

feature_names = xtrain.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))

Sex: 28.377591527551807
Pclass: 17.450379813673287
Parch: 10.276200044515498
Embarked: 8.761954037905873
Cabin: 8.281577549519369
SibSp: 7.950157281933983
Age: 7.842375602284014
Ticket: 5.620556803330715
Fare: 5.4392073392855105
PassengerId: 0.0
Name: 0.0


Сохранение модели

In [ ]:
# сохраняем модель
# model.save_model('catboost_model.dump')

# загружаем сохраненную модель
# model.load_model('catboost_model.dump');